# config: YAML config loader

Tiny helper used by every run notebook: `load_config("configs/train.yaml")` gives nested
`SimpleNamespace`s (`cfg.cnn.lr` style access) and `to_dict(cfg)` converts back (for
saving the config next to a checkpoint).

Paths inside the configs are relative to `gambit_vision/`. Notebooks live in
`gambit_vision/notebooks/`, so `GV_ROOT` points one level up and `gv_path()` resolves a
config path against it, whatever the current directory is.

Milestone: shared plumbing (M1 to M5).

**TODO**
- [ ] nothing required; add helpers here if several notebooks need them.

In [ ]:
from __future__ import annotations

from pathlib import Path
from types import SimpleNamespace
from typing import Any

import yaml

## Package root
`GV_ROOT` is `gambit_vision/`, found from the current directory (the notebooks folder when a
notebook runs, or `gambit_vision/` itself if a run notebook already did `%cd ..`).

In [ ]:
def _find_gv_root() -> Path:
    here = Path.cwd().resolve()
    for p in (here, *here.parents):
        if (p / "configs" / "train.yaml").exists() and (p / "notebooks").is_dir():
            return p
    return here.parent if here.name == "notebooks" else here


GV_ROOT = _find_gv_root()
REPO_ROOT = GV_ROOT.parent


def gv_path(p: str | Path) -> Path:
    """Resolve a config path (relative to gambit_vision/) to an absolute path."""
    p = Path(p)
    return p if p.is_absolute() else (GV_ROOT / p)

## Loading and saving configs

In [ ]:
def _to_ns(obj: Any) -> Any:
    if isinstance(obj, dict):
        return SimpleNamespace(**{k: _to_ns(v) for k, v in obj.items()})
    if isinstance(obj, list):
        return [_to_ns(v) for v in obj]
    return obj


def to_dict(obj: Any) -> Any:
    """Inverse of load_config's conversion (for saving the config next to a checkpoint)."""
    if isinstance(obj, SimpleNamespace):
        return {k: to_dict(v) for k, v in vars(obj).items()}
    if isinstance(obj, list):
        return [to_dict(v) for v in obj]
    return obj


def load_config(path: str | Path) -> SimpleNamespace:
    """Load a YAML file into nested SimpleNamespaces.

    Relative paths are resolved against gambit_vision/ (GV_ROOT). Paths *inside* the config
    stay relative to gambit_vision/; run notebooks `%cd ..` so they resolve.
    """
    with open(gv_path(path)) as f:
        return _to_ns(yaml.safe_load(f) or {})

## Self-check

In [ ]:
_cfg = load_config("configs/train.yaml")
assert _cfg.cnn.num_classes == 13 and _cfg.cnn.arch == "resnet18"
assert to_dict(_cfg)["cnn"]["input_size"] == [200, 100]
assert _to_ns({"a": [{"b": 1}]}).a[0].b == 1
assert load_config("configs/camera.yaml").crop.size_wh == [100, 200]
print("config OK, GV_ROOT =", GV_ROOT)